# Aggregating to Decrease Size

In [1]:
library(tidyverse)

#####
# Loading in _Post_-Covid Data
#####
dir = "../data/od-data-post-covid.csv"
od_data <- read_csv(dir, 
        col_names = c(
            "date", 
            "hour", 
            "origin",
            "destination",
            "ridership"
        ))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 28310492 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): date, hour, origin, destination, ridership

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [2]:
format(object.size(od_data), units = "GB")

[1] "1.1 Gb"

Taken directly from the `filter-and-add-features.r` file

In [4]:
library(tis)
library(baseballr)
library(hoopR)

add_day_of_week_indicator <- function(df) {
    # Function to add a day of week indicator to the dataframe
    df <- df |> mutate(day = case_when(wday(date) %in% seq(2, 6) ~ "weekday",
                                            wday(date) == 1 ~ "sunday",
                                            wday(date) == 7 ~ "saturday")
                        )
    return(df)
}

add_line_indicator <- function(df) {
    # Function to add a line indicator for each BART origin/destination line
    # List of all BART Lines
    bart_lines <- list(
    green = c("BERY", "MLPT", "WARM", "FRMT", "UCTY", "SHAY", "HAYW", "BAYF",
            "SANL", "COLS", "FTVL", "LAKE", "WOAK", "EMBR", "MONT", "POWL", 
            "CIVC", "16TH", "24TH", "GLEN", "BALB", "DALY"),
            
    red = c("RICH", "DELN", "PLZA", "NBRK", "DBRK", "ASHB", "MCAR", "19TH", 
          "12TH", "WOAK", "EMBR", "MONT", "POWL", "CIVC", "16TH", "24TH", 
          "GLEN", "BALB", "DALY", "COLM", "SSAN", "SBRN", "SFIA", "MLBR"),
          
    yellow = c("ANTC", "PCTR", "PITT", "NCON", "CONC", "PHIL", "WCRK", "LAFY", 
             "ORIN", "ROCK", "MCAR", "19TH", "12TH", "WOAK", "EMBR", "MONT", 
             "POWL", "CIVC", "16TH", "24TH", "GLEN", "BALB", "DALY", "COLM", 
             "SSAN", "SBRN", "SFIA", "MLBR"),
             
    blue = c("DUBL", "WDUB", "CAST", "BAYF", "SANL", "COLS", "FTVL", "LAKE", 
           "WOAK", "EMBR", "MONT", "POWL", "CIVC", "16TH", "24TH", "GLEN", 
           "BALB", "DALY"),
           
    orange = c("BERY", "MLPT", "WARM", "FRMT", "UCTY", "SHAY", "HAYW", "BAYF", 
             "SANL", "COLS", "FTVL", "LAKE", "12TH", "19TH", "MCAR", "ASHB", 
             "DBRK", "NBRK", "PLZA", "DELN", "RICH")
             )

    
    for (color in names(bart_lines)) {
        origin_column <- paste0("origin_", color)
        destination_column <- paste0("destination_", color)
        df <- df %>%
        mutate(
        !!origin_column := origin %in% bart_lines[[color]],
        !!destination_column := destination %in% bart_lines[[color]]
        )
    }
    return(df)

}

add_holiday_indicator <- function(df) {
    # Function to add a holiday indicator to the dataframe
    holiday_dates <- as.Date(as.character(holidays(2018:2025, board = TRUE)), format = "%Y%m%d")
    df <- df %>% mutate(is_holiday = date %in% holiday_dates)
    return(df)
}

add_baseball_game_indicator <- function(df) {
    # Function to add a baseball game indicator to the dataframe

    # Gather Dates of Home Games for Giants and A's from 2018-2025
    get_home_games <- function(team_id, year) {
        mlb_schedule(year) %>%
        filter(teams_home_team_id == team_id) %>%
        pull(date) %>%
        as.Date()
    }
    years <- 2018:2025
    giants_home_dates <- map(years, ~get_home_games(137, .x)) %>% list_c()
    as_home_dates <- map(years, ~get_home_games(133, .x)) %>% list_c()

    df <- df %>%
    mutate(
      is_giants_home = date %in% giants_home_dates,
      is_as_home = date %in% as_home_dates
      )
    return(df)
}

add_warriors_game_indicator <- function(df) {
    # Function to add a warriors game indicator to the dataframe

    # Gather Dates of Home Games for Warriors from 2018-2025
    warriors_home_dates <- load_nba_schedule(2018:2025) %>%
    filter(home_id == 9) %>%
    pull(game_date) %>%
    as.Date()

    df <- df %>%
    mutate(
      warriors_at_coliseum = (date %in% warriors_home_dates) & (date < as.Date("2019-10-05")),
      warriors_at_chase = (date %in% warriors_home_dates) & (date >= as.Date("2019-10-05")) # Warriors first game (pre-season) at Chase
      )
    return(df)
}

add_season_indicator <- function(df) {
    # Function to add a season indicator to the dataframe
    df <- df %>% mutate(season = case_when(month(date) %in% c(12, 1, 2) ~ "winter",
                                            month(date) %in% c(3, 4, 5) ~ "spring",
                                            month(date) %in% c(6, 7, 8) ~ "summer",
                                            month(date) %in% c(9, 10, 11) ~ "fall")
                        )
    return(df)
}

filter_operational_hours <- function(df) {
    # Function to filter the dataframe to only include operational hours (5am-1am)
    df <- df %>% filter((day == "weekday") & !(hour %in% 0:4) |
                        (day == "saturday") & !(hour %in% 0:5) | 
                        (day == "sunday") & !(hour %in% 0:7))
    return(df)
}

add_all_features <- function(df) {
    # Function to filter the dataframe and add features
    df <- df %>%
    add_line_indicator() %>%
    add_holiday_indicator() %>%
    add_baseball_game_indicator() %>%
    add_warriors_game_indicator() %>%
    add_season_indicator()
    return(df)
}
